# Câu a: Tìm tập phổ biến bằng Apriori và ghi nhận thời gian xử lý

Phần 2 – Bài 6 (trang 45): Groceries – 9.835 giao dịch, 169 mặt hàng.

**Về khoảng min_sup 0.01% – 0.1%:** với 9.835 giao dịch, min_sup = 0.01% tương đương support count ≥ 1, tức là *mọi* tập con của *mọi* giao dịch đều phổ biến (giao dịch dài nhất có 32 mặt hàng → tới 2³² tập con), nên không thuật toán nào chạy hết được. Thử thực tế trên máy 8 GB RAM: Apriori bị tràn bộ nhớ từ min_sup = 0.04% trở xuống. Vì vậy bài này khảo sát **min_sup từ 0.1% giảm dần tới 0.05%** (bước 0.01%) cho cả hai thuật toán, và chạy thêm FP-Growth ở 0.04%, 0.03%, 0.02% để thấy FP-Growth vẫn xử lý được khi Apriori đã không chạy nổi.

In [1]:
SV = "Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT"
DATA = "../../data/"  # thư mục chứa dữ liệu
print(SV)

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT


In [2]:
import time
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth

---
## Thực hiện yêu cầu

In [3]:
print(SV)
# Mỗi dòng là một giao dịch, các mặt hàng cách nhau bởi dấu phẩy (số mặt hàng mỗi dòng khác nhau)
with open(DATA + 'groceries.csv', encoding='utf-8') as f:
    transactions = [line.strip().split(',') for line in f if line.strip()]

te = TransactionEncoder()
df = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)

SUPPORTS = [0.001, 0.0009, 0.0008, 0.0007, 0.0006, 0.0005]   # 0.1% -> 0.05%
SUPPORTS_FP_EXTRA = [0.0004, 0.0003, 0.0002]                 # chỉ FP-Growth chạy được
print('Số giao dịch:', len(transactions))
print('Số mặt hàng :', df.shape[1])
df.head()

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT
Số giao dịch: 9835
Số mặt hàng : 169


,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,baby food,bags,baking powder,bathroom cleaner,beef,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [4]:
res_ap = []
for s in SUPPORTS:
    t0 = time.perf_counter()
    fi = apriori(df, min_support=s, use_colnames=True)
    res_ap.append({'min_sup (%)': s * 100, 'số tập phổ biến': len(fi), 'thời gian Apriori (s)': round(time.perf_counter() - t0, 3)})
res_ap = pd.DataFrame(res_ap)
res_ap

,min_sup (%),số tập phổ biến,thời gian Apriori (s)
0,0.10,13492,1.709
1,0.09,16188,1.894
2,0.08,19979,2.168
3,0.07,25570,2.842
4,0.06,34353,3.634
5,0.05,49580,5.466


In [5]:
# Ví dụ các tập phổ biến nhiều phần tử nhất ở min_sup = 0.05%
fi = apriori(df, min_support=0.0005, use_colnames=True)
fi['length'] = fi['itemsets'].apply(len)
print(fi['length'].value_counts().sort_index())
fi.sort_values(['length', 'support'], ascending=False).head(10)

length
1      164
2     4782
3    19226
4    18597
5     5988
6      785
7       38
Name: count, dtype: int64


,support,itemsets,length
49545,0.000610,"frozenset({other vegetables, root vegetables, ...",7
49548,0.000610,"frozenset({other vegetables, root vegetables, ...",7
49555,0.000610,"frozenset({other vegetables, root vegetables, ...",7
49561,0.000610,"frozenset({other vegetables, root vegetables, ...",7
49573,0.000610,"frozenset({oil, fruit/vegetable juice, other v...",7
49542,0.000508,"frozenset({root vegetables, other vegetables, ...",7
49543,0.000508,"frozenset({root vegetables, beef, yogurt, saus...",7
49544,0.000508,"frozenset({other vegetables, root vegetables, ...",7
49546,0.000508,"frozenset({other vegetables, root vegetables, ...",7
49547,0.000508,"frozenset({other vegetables, butter, yogurt, d...",7


**Nhận xét:**

- Groceries có 9.835 giao dịch, 169 mặt hàng; giao dịch dài nhất 32 mặt hàng.
- Khi min_sup giảm từ 0,1% xuống 0,05%, số tập phổ biến tăng gần gấp 4 (13.492 → 49.580), và thời gian Apriori cũng tăng khoảng 3–4 lần (từ khoảng 1,5–2 giây lên khoảng 6 giây; số đo cụ thể thay đổi theo máy).
- Ở min_sup = 0,05% đã xuất hiện các tập phổ biến dài tới 7 mặt hàng, phần lớn là các tập 3–4 phần tử (≈19.000 tập mỗi loại).
- Ở mức 0,04% trở xuống, Apriori sinh quá nhiều ứng viên và bị tràn bộ nhớ (8 GB) nên không ghi nhận được thời gian.